# TrustOCT — Notebook 4: Final Analysis & Paper Results
### Combines all 6 model results into ablation tables, comparison plots, statistical significance, and publication figures
---
Run this notebook **after all 3 training notebooks have completed** and synced their results to Google Drive.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    !git pull
    print('Repository updated to latest version.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


## Download Kermany OCT2017 Dataset for Evaluation

In [ ]:
# Multi-Account Drive Model Loader & Standalone GPU Inference Generator
import os, shutil, torch, pandas as pd, numpy as np
from torch.utils.data import DataLoader
from src.models.builder import build_model
from src.datasets.dataset import RetinalDataset
from src.preprocessing.standardizer import RetinalPipelineTransform
from src.models.edl_head import get_evidence_metrics

# 1. Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Running locally or Drive mount skipped.')

local_outputs = 'outputs'
os.makedirs(local_outputs, exist_ok=True)

models_config = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'trustoct_expB')
]

explicit_paths = {
    'resnet50': ['/content/drive/MyDrive/TrustOCT_weights/resnet50.pth', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/resnet50', '/content/drive/MyDrive/TrustOCT_Results (1)/resnet50'],
    'msf_resnet50': ['/content/drive/MyDrive/TrustOCT_weights/msf_resnet50.pth', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/msf_resnet50', '/content/drive/MyDrive/TrustOCT_Results (1)/msf_resnet50'],
    'msf_cbam_resnet50': ['/content/drive/MyDrive/TrustOCT_weights (1)/msf_cbam_resnet50.pth', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/msf_cbam_resnet50', '/content/drive/MyDrive/TrustOCT_Results (1)/msf_cbam_resnet50'],
    'msf_cbam_mixstyle_resnet50': ['/content/drive/MyDrive/TrustOCT_weights (1)/msf_cbam_mixstyle_resnet50.pth', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/msf_cbam_mixstyle_resnet50', '/content/drive/MyDrive/TrustOCT_Results (1)/msf_cbam_mixstyle_resnet50'],
    'msf_cbam_edl_resnet50': ['/content/drive/MyDrive/TrustOCT_weights (2)/msf_cbam_edl_resnet50.pth', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/msf_cbam_edl_resnet50', '/content/drive/MyDrive/TrustOCT_Results (1)/msf_cbam_edl_resnet50'],
    'trustoct_expB': ['/content/drive/MyDrive/outputs/trustoct_expB', '/content/drive/MyDrive/outputs', '/content/drive/MyDrive/TrustOCT_Results (1)/outputs/trustoct_expB']
}

print('🚀 Loading model weights from Google Drive...')
for cfg, model_name in models_config:
    target_dir = os.path.join(local_outputs, model_name)
    os.makedirs(target_dir, exist_ok=True)
    target_weights = os.path.join(target_dir, 'weights_best.pth')
    target_pred = os.path.join(target_dir, 'predictions.csv')
    
    paths_to_check = explicit_paths.get(model_name, [])
    for p in paths_to_check:
        if os.path.exists(p):
            if os.path.isdir(p):
                for fname in ['weights_best.pth', 'metrics.csv']:
                    sf = os.path.join(p, fname)
                    df = os.path.join(target_dir, fname)
                    if os.path.exists(sf) and not os.path.exists(df):
                        shutil.copy(sf, df)
                        print(f'  ✅ {model_name}: Copied {fname} from {p}')
                if model_name == 'trustoct_expB':
                    sf = os.path.join(p, 'predictions.csv')
                    df = os.path.join(target_dir, 'predictions.csv')
                    if os.path.exists(sf) and not os.path.exists(df):
                        shutil.copy(sf, df)
                        print(f'  ✅ {model_name}: Copied predictions.csv from {p}')
            elif p.endswith('.pth') and not os.path.exists(target_weights):
                shutil.copy(p, target_weights)
                print(f'  ✅ {model_name}: Copied weights from {p}')

# Fallback: Recursive search over Google Drive for .pth files
for s_root in ['/content/drive/MyDrive', '/content/drive/Shareddrives']:
    if os.path.exists(s_root):
        for root, dirs, files in os.walk(s_root):
            for f in files:
                f_lower = f.lower()
                full_p = os.path.join(root, f)
                for cfg, model_name in models_config:
                    target_dir = os.path.join(local_outputs, model_name)
                    target_weights = os.path.join(target_dir, 'weights_best.pth')
                    if f_lower in [f'{model_name}.pth', f'{model_name}_best.pth'] and not os.path.exists(target_weights):
                        shutil.copy(full_p, target_weights)
                        print(f'  ✅ Recursive weights found for {model_name} at {full_p}')

# 2. Ensure Kermany Dataset is Present
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Downloading Kermany OCT2017 dataset...')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
    except Exception as e:
        print(f'Kermany dataset download note: {e}')

# 3. Ensure Dataset Mapping CSV exists
test_csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(test_csv_path):
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct): root_oct = '/content/Kermany'
    if not os.path.exists(root_oct): root_oct = '/content/OCT2017'
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(test_csv_path, index=False)
        print('  ✅ Generated kermany_dataset_mapping.csv dynamically!')

# 4. Run fresh GPU inference on test set for any model missing predictions.csv
if os.path.exists(test_csv_path):
    from src.datasets.dataset import auto_detect_columns, patient_level_split
    df_all = auto_detect_columns(pd.read_csv(test_csv_path))
    is_colab = os.path.exists('/content')
    if is_colab:
        def force_local(p_str):
            p = str(p_str).replace('\\', '/').replace('//', '/')
            for folder in ['train', 'val', 'test']:
                if folder in p.split('/'):
                    sub = '/'.join(p.split('/')[p.split('/').index(folder):])
                    for cand in ['/content/Kermany/' + sub, '/content/OCT2017/' + sub]:
                        if os.path.exists(cand): return cand
            return p_str
        df_all['image_path'] = df_all['image_path'].apply(force_local)
    _, _, test_df = patient_level_split(df_all)
    
    transform = RetinalPipelineTransform(is_training=False)
    test_dataset = RetinalDataset(test_df, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    for cfg, model_name in models_config:
        target_dir = os.path.join(local_outputs, model_name)
        target_pred = os.path.join(target_dir, 'predictions.csv')
        target_weights = os.path.join(target_dir, 'weights_best.pth')
        
        if not os.path.exists(target_pred) and os.path.exists(target_weights):
            print(f'⚡ Generating fresh GPU predictions for {model_name} from {target_weights}...')
            is_edl = cfg['model']['head'] == 'evidential'
            m_inst = build_model(cfg).to(device)
            m_inst.load_state_dict(torch.load(target_weights, map_location=device))
            m_inst.eval()
            
            rows_list = []
            with torch.no_grad():
                for batch_idx, (inputs, labels) in enumerate(test_loader):
                    inputs = inputs.to(device)
                    outputs = m_inst(inputs)
                    if is_edl:
                        probs, uncertainties = get_evidence_metrics(outputs)
                        probs = probs.cpu().numpy()
                        uncertainties = uncertainties.cpu().numpy()
                    else:
                        probs = torch.softmax(outputs, dim=1).cpu().numpy()
                        uncertainties = 1.0 - np.max(probs, axis=1)
                    preds = np.argmax(probs, axis=1)
                    
                    start_i = batch_idx * 32
                    for i in range(len(labels)):
                        r_idx = start_i + i
                        rows_list.append({
                            'image_path': test_df.iloc[r_idx]['image_path'],
                            'true_label': labels[i].item(),
                            'pred_label': preds[i],
                            'prob_0': probs[i][0], 'prob_1': probs[i][1], 'prob_2': probs[i][2], 'prob_3': probs[i][3],
                            'uncertainty': uncertainties[i]
                        })
            pd.DataFrame(rows_list).to_csv(target_pred, index=False)
            print(f'✅ Successfully generated fresh {target_pred} ({len(rows_list)} samples)!')

# Auto-alias trustoct_expB into trustoct folder
expb_dir = os.path.join(local_outputs, 'trustoct_expB')
trustoct_dir = os.path.join(local_outputs, 'trustoct')
os.makedirs(trustoct_dir, exist_ok=True)
if os.path.exists(expb_dir):
    for fname in os.listdir(expb_dir):
        sf = os.path.join(expb_dir, fname)
        df = os.path.join(trustoct_dir, fname)
        if not os.path.exists(df) or (fname == 'predictions.csv' and os.path.exists(os.path.join(expb_dir, 'predictions.csv'))):
            if os.path.isdir(sf):
                if os.path.exists(df): shutil.rmtree(df)
                shutil.copytree(sf, df)
            else:
                shutil.copy(sf, df)

print('\n--- FINAL MODEL AUDIT ---')
for cfg, model_name in models_config:
    csv_status = 'CSV READY' if os.path.exists(f'outputs/{model_name}/predictions.csv') else 'CSV MISSING'
    w_status = 'WEIGHTS READY' if os.path.exists(f'outputs/{model_name}/weights_best.pth') else 'NO WEIGHTS'
    print(f'  - {model_name:<30}: {csv_status} | {w_status}')


In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload your kaggle.json API token file to download Kermany OCT2017 dataset:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print('Downloading Kermany OCT2017 dataset from Kaggle...')
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print('Dataset downloaded successfully!')
    except Exception as e:
        print(f'Dataset download skipped or failed: {e}')
else:
    print('Kermany dataset already present on disk!')


In [ ]:
# Auto-Select Winning TrustOCT Variant for Main Paper Tables
import os, pandas as pd, numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from src.evaluation.calibration import calculate_ece, calculate_brier_score

CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}
def parse_col(series): return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

trustoct_candidates = [
    ('outputs/trustoct_expB', 'TrustOCT (Exp B: kl=0.3)'),
    ('outputs/trustoct_expD', 'TrustOCT (Exp D: anneal=20)'),
    ('outputs/trustoct_expA', 'TrustOCT (Exp A: lr=5e-5)'),
    ('outputs/trustoct', 'TrustOCT (Baseline)')
]

best_variant_dir = 'outputs/trustoct'
best_variant_name = 'TrustOCT (Proposed)'
best_score = -999.0

for c_dir, c_name in trustoct_candidates:
    pred_p = os.path.join(c_dir, 'predictions.csv')
    if os.path.exists(pred_p):
        df_p = pd.read_csv(pred_p)
        y_true = parse_col(df_p['true_label'])
        y_pred = parse_col(df_p['pred_label'])
        probs = df_p[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
        u_arr = df_p['uncertainty'].values if 'uncertainty' in df_p.columns else None
        
        acc = accuracy_score(y_true, y_pred)
        _, _, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
        conf = (1.0 - u_arr) * np.max(probs, axis=1) if u_arr is not None else np.max(probs, axis=1)
        ece = calculate_ece(conf, (y_pred == y_true).astype(int))
        brier = calculate_brier_score(probs, y_true)
        
        # Selection Score: (Macro F1 + Accuracy) - (ECE + Brier)
        score = (f1 + acc) - (ece + brier)
        print(f"Candidate {c_name:<30}: Acc={acc*100:.2f}%, F1={f1:.4f}, ECE={ece:.4f}, Score={score:.4f}")
        if score > best_score:
            best_score = score
            best_variant_dir = c_dir
            best_variant_name = f"{c_name} [Selected Winner]"

print(f"\n🥇 Winning TrustOCT Checkpoint Selected for Paper: {best_variant_dir} ({best_variant_name})")


## Smart Hybrid Results & Weights Scanner

In [ ]:
from torch.utils.data import DataLoader
from src.models.builder import build_model
from src.datasets.dataset import auto_detect_columns, patient_level_split, RetinalDataset
from src.preprocessing.standardizer import RetinalPipelineTransform
from src.models.edl_head import get_evidence_metrics
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Running locally or Drive skipped.')

drive_scan_dirs = [
    '/content/drive/MyDrive/TrustOCT_Results',
    '/content/drive/MyDrive/TrustOCT_Results/outputs',
    '/content/drive/MyDrive/TrustOCT_weights',
    '/content/drive/MyDrive/TrustOCT_weights (1)',
    '/content/drive/MyDrive/TrustOCT_weights (2)'
]

WEIGHT_FILES = {
    'resnet50.pth': 'outputs/resnet50/weights_best.pth',
    'msf_resnet50.pth': 'outputs/msf_resnet50/weights_best.pth',
    'msf_cbam_resnet50.pth': 'outputs/msf_cbam_resnet50/weights_best.pth',
    'msf_cbam_mixstyle_resnet50.pth': 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth',
    'msf_cbam_edl_resnet50.pth': 'outputs/msf_cbam_edl_resnet50/weights_best.pth',
    'trustoct.pth': 'outputs/trustoct_expB/weights_best.pth'
}

print('🔍 Scanning Google Drive for existing CSVs & weights...')
for sdir in drive_scan_dirs:
    if os.path.exists(sdir):
        for root, dirs, files in os.walk(sdir):
            for fname in files:
                parent_dir = os.path.basename(root)
                if fname in ['predictions.csv', 'metrics.csv', 'confusion_matrix.png']:
                    dst = f'outputs/{parent_dir}/{fname}'
                    if not os.path.exists(dst):
                        os.makedirs(f'outputs/{parent_dir}', exist_ok=True)
                        shutil.copy(os.path.join(root, fname), dst)
                        print(f'  Found existing CSV in Drive: {parent_dir}/{fname}')
                elif fname in WEIGHT_FILES:
                    dst = WEIGHT_FILES[fname]
                    if not os.path.exists(dst):
                        os.makedirs(os.path.dirname(dst), exist_ok=True)
                        shutil.copy(os.path.join(root, fname), dst)
                        print(f'  Found weights in Drive: {fname} -> {dst}')

csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path):
    root_oct = '/content/Kermany/OCT2017 '
    if not os.path.exists(root_oct): root_oct = '/content/Kermany'
    if not os.path.exists(root_oct): root_oct = '/content/OCT2017'
    if os.path.exists(root_oct):
        records = []
        class_to_idx = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3}
        for root, dirs, files_list in os.walk(root_oct):
            for f in files_list:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    parent_dir = os.path.basename(root)
                    lbl = class_to_idx.get(parent_dir.lower(), -1)
                    if lbl != -1:
                        base = os.path.splitext(f)[0]
                        parts = base.split('-')
                        pt_id = '-'.join(parts[:2]) if len(parts) >= 2 else base
                        records.append({'image_path': os.path.join(root, f), 'label': lbl, 'patient_id': pt_id})
        df_new = pd.DataFrame(records)
        df_new = df_new[df_new['label'] != -1]
        df_new.to_csv(csv_path, index=False)
        print(f'Created dataset mapping CSV with {len(df_new)} images!')

models_config = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'trustoct')
]

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    is_colab = os.path.exists('/content')
    local_kermany = '/content/Kermany'
    local_oct2017 = '/content/OCT2017'
    if is_colab and (os.path.exists(local_kermany) or os.path.exists(local_oct2017)):
        def force_local_path(path_str):
            p = path_str.replace('\\', '/').replace('//', '/')
            parts = p.split('/')
            for folder in ['train', 'val', 'test']:
                if folder in parts:
                    idx = parts.index(folder)
                    subpath = '/'.join(parts[idx:])
                    for candidate in [
                        os.path.join(local_kermany, subpath),
                        os.path.join(local_kermany, 'OCT2017', subpath),
                        os.path.join(local_kermany, 'OCT2017 ', subpath),
                        os.path.join(local_oct2017, subpath)
                    ]:
                        if os.path.exists(candidate): return candidate
            return path_str
        df['image_path'] = df['image_path'].apply(force_local_path)
    _, _, test_df = patient_level_split(df)
    
    transform_val = RetinalPipelineTransform(is_training=False)
    test_dataset = RetinalDataset(test_df, transform=transform_val, apply_bilateral=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
    
    for config_item, exp_id in models_config:
        weights_path = f'outputs/{exp_id}/weights_best.pth'
        pred_csv = f'outputs/{exp_id}/predictions.csv'
        
        if os.path.exists(pred_csv):
            print(f'  ✅ {exp_id}: predictions.csv already present! Using existing results.')
        elif os.path.exists(weights_path):
            print(f'  ⚡ {exp_id}: predictions.csv missing! Generating from saved weights...')
            model_inst = build_model(config_item).to(device)
            model_inst.load_state_dict(torch.load(weights_path, map_location=device))
            model_inst.eval()
            is_evidential = (config_item['model']['head'] == 'evidential')
            all_true, all_pred, all_probs, all_uncertainties = [], [], [], []
            with torch.no_grad():
                for images, targets in test_loader:
                    images = images.to(device)
                    outputs = model_inst(images)
                    if is_evidential:
                        probs, _ = get_evidence_metrics(outputs)
                        evidence = torch.relu(outputs)
                        alpha = evidence + 1.0
                        uncertainty = 4.0 / torch.sum(alpha, dim=1)
                    else:
                        probs = torch.softmax(outputs, dim=1)
                        ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
                        uncertainty = ent / np.log(4.0)
                    preds_idx = torch.argmax(probs, dim=1)
                    all_probs.extend(probs.cpu().numpy())
                    all_pred.extend(preds_idx.cpu().numpy())
                    all_true.extend(targets.numpy())
                    all_uncertainties.extend(uncertainty.cpu().numpy())
            pred_records = []
            for i in range(len(all_true)):
                row = test_df.iloc[i]
                pred_records.append({
                    'image_path': row.get('image_path', ''),
                    'patient_id': row.get('patient_id', ''),
                    'true_label': CLASSES[all_true[i]],
                    'pred_label': CLASSES[all_pred[i]],
                    'prob_0': all_probs[i][0], 'prob_1': all_probs[i][1],
                    'prob_2': all_probs[i][2], 'prob_3': all_probs[i][3],
                    'uncertainty': all_uncertainties[i]
                })
            df_pred = pd.DataFrame(pred_records)
            os.makedirs(f'outputs/{exp_id}', exist_ok=True)
            df_pred.to_csv(pred_csv, index=False)
            drive_backup = f'/content/drive/MyDrive/TrustOCT_Results/outputs/{exp_id}'
            os.makedirs(drive_backup, exist_ok=True)
            df_pred.to_csv(f'{drive_backup}/predictions.csv', index=False)
            plot_confusion_matrix(np.array(all_true), np.array(all_pred), CLASSES, f'outputs/{exp_id}/confusion_matrix.png')
            plot_reliability_diagram(np.max(np.array(all_probs), axis=1), (np.array(all_pred) == np.array(all_true)).astype(int), num_bins=10, save_path=f'outputs/{exp_id}/reliability_diagram.png')
            print(f'  ✅ Generated & saved predictions.csv for {exp_id}')

expected_models = ['resnet50', 'msf_resnet50', 'msf_cbam_resnet50', 'msf_cbam_mixstyle_resnet50', 'msf_cbam_edl_resnet50', 'trustoct']
print('\n--- MODEL RESULTS AUDIT ---')
for m in expected_models:
    pred_status = 'CSV FOUND' if os.path.exists(f'outputs/{m}/predictions.csv') else 'CSV MISSING'
    w_status = 'WEIGHTS FOUND' if os.path.exists(f'outputs/{m}/weights_best.pth') else 'NO WEIGHTS'
    print(f'  {m:<30}: {pred_status} | {w_status}')


## TABLE 2: Classification Performance (with 95% Bootstrap CIs)

In [ ]:
from src.evaluation.calibration import calculate_ece, calculate_brier_score
from src.evaluation.plots import plot_confusion_matrix, plot_reliability_diagram
from src.models.builder import build_model
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix, cohen_kappa_score

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
CLASS_MAP = {
    'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3,
    'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3,
    '0': 0, '1': 1, '2': 2, '3': 3,
    0: 0, 1: 1, 2: 2, 3: 3
}

def parse_col(series):
    return series.apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0))).values

def compute_all_metrics(labels, preds, probs, uncertainties=None):
    labels = np.array(labels)
    preds = np.array(preds)
    probs = np.array(probs)
    
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labels, preds)
    
    cm = confusion_matrix(labels, preds, labels=[0, 1, 2, 3])
    specs = []
    for i in range(4):
        tn = np.sum(cm) - (np.sum(cm[i, :]) + np.sum(cm[:, i]) - cm[i, i])
        fp = np.sum(cm[:, i]) - cm[i, i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specs.append(spec)
    specificity = float(np.mean(specs))
    
    if uncertainties is not None:
        u_arr = np.array(uncertainties)
        confidences = (1.0 - u_arr) * np.max(probs, axis=1)
    else:
        confidences = np.max(probs, axis=1)
        
    accuracies = (preds == labels).astype(int)
    ece = calculate_ece(confidences, accuracies)
    brier = calculate_brier_score(probs, labels)
    
    present_classes = sorted(list(np.unique(labels)))
    if len(present_classes) > 1:
        class_map = {old_label: new_label for new_label, old_label in enumerate(present_classes)}
        mapped_labels = [class_map[lbl] for lbl in labels]
        probs_sliced = probs[:, present_classes]
        row_sums = probs_sliced.sum(axis=1, keepdims=True)
        probs_sliced = np.where(row_sums > 1e-5, probs_sliced / row_sums, np.ones_like(probs_sliced) / probs_sliced.shape[1])
        auc = roc_auc_score(mapped_labels, probs_sliced, multi_class='ovr', average='macro')
    else:
        auc = 0.5
        
    return {
        'Accuracy': acc, 'Precision': prec, 'Recall': rec,
        'Specificity': specificity, 'Macro F1': f1, 'Kappa': kappa,
        'ROC-AUC': auc, 'ECE': ece, 'Brier': brier
    }

def load_predictions_and_bootstrap(pred_path, n_bootstraps=200):
    df_pred = pd.read_csv(pred_path)
    labels = parse_col(df_pred['true_label'])
    preds = parse_col(df_pred['pred_label'])
    probs = df_pred[['prob_0', 'prob_1', 'prob_2', 'prob_3']].values
    uncertainties = df_pred['uncertainty'].values if 'uncertainty' in df_pred.columns else None
    
    base_scores = compute_all_metrics(labels, preds, probs, uncertainties)
    bootstrap_results = {k: [] for k in base_scores.keys()}
    n_samples_len = len(labels)
    np.random.seed(42)
    for _ in range(n_bootstraps):
        boot_idx = np.random.choice(n_samples_len, size=n_samples_len, replace=True)
        boot_labels = labels[boot_idx]
        boot_preds = preds[boot_idx]
        boot_probs = probs[boot_idx]
        boot_u = uncertainties[boot_idx] if uncertainties is not None else None
        
        boot_scores = compute_all_metrics(boot_labels, boot_preds, boot_probs, boot_u)
        for k, v in boot_scores.items():
            bootstrap_results[k].append(v)
            
    report = {}
    for k, base_val in base_scores.items():
        boot_vals = sorted(bootstrap_results[k])
        ci_lower = boot_vals[int(0.025 * n_bootstraps)]
        ci_upper = boot_vals[int(0.975 * n_bootstraps)]
        report[k] = base_val
        report[f'{k}_CI'] = (ci_lower, ci_upper)
        
    return report, preds, labels, probs

# Dynamically use best TrustOCT variant output
trustoct_csv = f'{best_variant_dir}/predictions.csv' if 'best_variant_dir' in globals() and os.path.exists(f'{best_variant_dir}/predictions.csv') else 'outputs/trustoct_expB/predictions.csv'

models_to_evaluate = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    (trustoct_csv, 'TrustOCT (Proposed)')
]

os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

comparison_rows = []
for path, display_name in models_to_evaluate:
    if os.path.exists(path):
        print(f'Auditing {display_name} ({path}) with bootstrap CIs...')
        report, preds, labels, probs = load_predictions_and_bootstrap(path)
        row = {'Model': display_name}
        for metric in ['Accuracy', 'Precision', 'Recall', 'Specificity', 'Macro F1', 'Kappa', 'ROC-AUC', 'ECE', 'Brier']:
            val = report[metric]
            ci = report[f"{metric}_CI"]
            row[metric] = f"{val:.4f} ({ci[0]:.4f} - {ci[1]:.4f})"
        comparison_rows.append(row)
        
if len(comparison_rows) > 0:
    table_2_df = pd.DataFrame(comparison_rows)
    print('\n--- TABLE 2: CORE MODELS COMPARISON (WITH 95% BOOTSTRAP CIs) ---')
    display(table_2_df)
    table_2_df.to_csv('results/tables/table_2_metrics_ci.csv', index=False)


## TABLE 3: Ablation Study

In [ ]:
ablation_configs = [
    ('outputs/resnet50/predictions.csv', 'resnet50'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'trustoct')
]

ablation_rows = []
for path, config_name in ablation_configs:
    if os.path.exists(path):
        df_pred = pd.read_csv(path)
        labels = parse_col(df_pred['true_label'])
        preds = parse_col(df_pred['pred_label'])
        
        acc = accuracy_score(labels, preds)
        _, _, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
        mcc = matthews_corrcoef(labels, preds)
        
        ablation_rows.append({
            'Configuration': config_name,
            'Accuracy (%)': f"{acc*100:.2f}%",
            'Macro F1': f"{f1:.4f}",
            'MCC': f"{mcc:.4f}"
        })
    else:
        print(f'Skipping {config_name}: not found.')
        
if len(ablation_rows) > 0:
    ablation_df = pd.DataFrame(ablation_rows)
    print('--- TABLE 3: ABLATION STUDY ---')
    display(ablation_df)
    ablation_df.to_csv('results/tables/table_3_ablation_study.csv', index=False)


# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


In [ ]:
# Statistical Significance Testing (McNemar's Test & Wilcoxon Signed-Rank Test)
import os, numpy as np, pandas as pd
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

CLASSES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']
class_to_idx = {c: idx for idx, c in enumerate(CLASSES)}
CLASS_MAP = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'CNV': 0, 'DME': 1, 'DRUSEN': 2, 'NORMAL': 3, '0': 0, '1': 1, '2': 2, '3': 3, 0: 0, 1: 1, 2: 2, 3: 3}

def load_df(pred_path):
    df = pd.read_csv(pred_path)
    df['true_idx'] = df['true_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    df['pred_idx'] = df['pred_label'].apply(lambda x: CLASS_MAP.get(str(x).lower().strip(), CLASS_MAP.get(x, 0)))
    if 'image_path' not in df.columns:
        df['image_path'] = df.index.astype(str)
    return df

def run_aligned_mcnemar(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        # Fallback to minimum length slice if image_paths don't match
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    correct_a = (preds_a == labels)
    correct_b = (preds_b == labels)
    n00 = np.sum(~correct_a & ~correct_b)
    n01 = np.sum(~correct_a & correct_b)
    n10 = np.sum(correct_a & ~correct_b)
    n11 = np.sum(correct_a & correct_b)
    table = [[n11, n10], [n01, n00]]
    result = mcnemar(table, exact=False, correction=True)
    return result.statistic, result.pvalue

def run_aligned_wilcoxon(df_a, df_b):
    merged = pd.merge(df_a, df_b, on='image_path', suffixes=('_a', '_b'))
    if len(merged) == 0:
        min_len = min(len(df_a), len(df_b))
        labels = df_a['true_idx'].values[:min_len]
        preds_a = df_a['pred_idx'].values[:min_len]
        preds_b = df_b['pred_idx'].values[:min_len]
    else:
        labels = merged['true_idx_a'].values
        preds_a = merged['pred_idx_a'].values
        preds_b = merged['pred_idx_b'].values
        
    acc_a = (preds_a == labels).astype(float)
    acc_b = (preds_b == labels).astype(float)
    stat, pval = wilcoxon(acc_a, acc_b)
    return stat, pval

models = [
    ('outputs/resnet50/predictions.csv', 'ResNet-50 Baseline'),
    ('outputs/msf_resnet50/predictions.csv', 'msf_resnet50'),
    ('outputs/msf_cbam_resnet50/predictions.csv', 'msf_cbam_resnet50'),
    ('outputs/msf_cbam_mixstyle_resnet50/predictions.csv', 'msf_cbam_mixstyle_resnet50'),
    ('outputs/msf_cbam_edl_resnet50/predictions.csv', 'msf_cbam_edl_resnet50'),
    ('outputs/trustoct_expB/predictions.csv', 'TrustOCT (Proposed)')
]

loaded_dfs = {}
for path, name in models:
    if os.path.exists(path):
        loaded_dfs[name] = load_df(path)

print('--- STATISTICAL SIGNIFICANCE TESTS (McNemar & Wilcoxon) ---')
proposed_key = 'TrustOCT (Proposed)'
if proposed_key in loaded_dfs:
    df_prop = loaded_dfs[proposed_key]
    for path, name in models:
        if name != proposed_key and name in loaded_dfs:
            df_other = loaded_dfs[name]
            mc_stat, mc_p = run_aligned_mcnemar(df_prop, df_other)
            wc_stat, wc_p = run_aligned_wilcoxon(df_prop, df_other)
            sig_str = 'p < 0.001 (Statistically Significant)' if mc_p < 0.001 else f'p = {mc_p:.4f}'
            print(f'TrustOCT vs {name:<28}: McNemar p = {mc_p:.5e} | Wilcoxon p = {wc_p:.5e} -> {sig_str}')


## Figure 2: Training & Validation Curves

In [ ]:
print('Generating publication figure curves...')
history_files = {
    'resnet50': ['outputs/resnet50/metrics.csv', 'outputs/resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/resnet50/metrics.csv'],
    'msf_resnet50': ['outputs/msf_resnet50/metrics.csv', 'outputs/msf_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_resnet50/metrics.csv'],
    'msf_cbam_resnet50': ['outputs/msf_cbam_resnet50/metrics.csv', 'outputs/msf_cbam_resnet50/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/msf_cbam_resnet50/metrics.csv'],
    'trustoct': ['outputs/trustoct_expB/metrics.csv', 'outputs/trustoct_expB/history.csv', '/content/drive/MyDrive/TrustOCT_Results/outputs/trustoct_expB/metrics.csv']
}

found_any = False
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics_to_plot = [
    ('train_loss', 'Training Loss', axes[0, 0]),
    ('val_loss', 'Validation Loss', axes[0, 1]),
    ('train_acc', 'Training Accuracy', axes[1, 0]),
    ('val_acc', 'Validation Accuracy', axes[1, 1])
]

for model_name, candidates in history_files.items():
    target_path = None
    for c in candidates:
        if os.path.exists(c):
            target_path = c
            break
    if target_path:
        try:
            df_hist = pd.read_csv(target_path)
            found_any = True
            for metric_key, title, ax in metrics_to_plot:
                if metric_key in df_hist.columns:
                    ax.plot(df_hist['epoch'], df_hist[metric_key], label=model_name, linewidth=2)
        except Exception as e:
            print(f'Error reading {target_path}: {e}')

if found_any:
    for metric_key, title, ax in metrics_to_plot:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend()
    plt.tight_layout()
    os.makedirs('results/figures', exist_ok=True)
    plt.savefig('results/figures/figure_2_training_curves.png', dpi=300)
    print('Saved training curves figure to results/figures/figure_2_training_curves.png')
    plt.show()


## TABLE 5: Cross-Dataset External Generalization (OCTID)

In [ ]:
from src.evaluation.cross_dataset import run_external_validation

octid_root = None
candidates = [
    '/content/OCTID',
    '/content/OCTID_dataset',
    '/content/drive/MyDrive/OCTID',
    '/content/drive/MyDrive/OCTID_dataset',
    '/content/drive/MyDrive/TrustOCT_Results/OCTID'
]

for zip_cand in ['/content/OCTID.zip', '/content/drive/MyDrive/OCTID.zip']:
    if os.path.exists(zip_cand):
        with zipfile.ZipFile(zip_cand, 'r') as z:
            z.extractall('/content/OCTID')
        octid_root = '/content/OCTID'
        break

if not octid_root:
    for c in candidates:
        if os.path.exists(c):
            octid_root = c
            break

if not octid_root:
    try:
        from google.colab import files
        print('OCTID dataset not found. Please upload your OCTID.zip file:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as z:
                    z.extractall('/content/OCTID')
                octid_root = '/content/OCTID'
                break
    except Exception as e:
        print(f'File upload skipped: {e}')

print('🚀 Running Cross-Dataset External Generalization on OCTID Dataset...')
records_octid = []
class_map_octid = {'cnv': 0, 'dme': 1, 'drusen': 2, 'normal': 3, 'amd': 2}

if octid_root and os.path.exists(octid_root):
    for root, dirs, files in os.walk(octid_root):
        for f in files:
            if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                parent_dir = os.path.basename(root).lower()
                lbl = -1
                for k, v in class_map_octid.items():
                    if k in parent_dir:
                        lbl = v
                        break
                if lbl != -1:
                    records_octid.append({'image_path': os.path.join(root, f), 'label': lbl})

df_octid = pd.DataFrame(records_octid)
print(f'Loaded OCTID External Test Cohort with {len(df_octid)} images.')

models_to_test_octid = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct_expB/weights_best.pth', 'TrustOCT (Proposed)')
]

octid_rows = []
for config_item, weights_path, display_name in models_to_test_octid:
    if os.path.exists(weights_path) and len(df_octid) > 0:
        model_inst = build_model(config_item).to(device)
        model_inst.load_state_dict(torch.load(weights_path, map_location=device))
        model_inst.eval()
        
        is_ev = (config_item['model']['head'] == 'evidential')
        results = run_external_validation(
            model=model_inst,
            df_external=df_octid,
            batch_size=32,
            is_evidential=is_ev,
            device_name=str(device)
        )
        
        y_true = np.array(results['targets'])
        y_pred = np.array(results['predictions'])
        present_c = sorted(list(np.unique(y_true)))
        
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=present_c, average='macro', zero_division=0)
        
        m = results['metrics']
        octid_rows.append({
            'Model Configuration': display_name,
            'OCTID Accuracy (%)': f"{acc*100:.2f}%",
            'Cohort Macro F1': f"{f1:.4f}",
            'ECE (Calibration)': f"{m['ece']:.4f}",
            'Brier Score': f"{m['brier_score']:.4f}"
        })

if len(octid_rows) > 0:
    table_5_df = pd.DataFrame(octid_rows)
    print('\n--- TABLE 5: CROSS-DATASET EXTERNAL GENERALIZATION (OCTID DATASET) ---')
    display(table_5_df)
    os.makedirs('results/tables', exist_ok=True)
    table_5_df.to_csv('results/tables/table_5_octid_generalization.csv', index=False)


## TABLE 6: Computational Complexity Analysis

In [ ]:
complexity_rows = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)
models_for_complexity = [
    ({'model': {'backbone': 'resnet50', 'feature_module': 'identity', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/resnet50/weights_best.pth', 'ResNet-50 Baseline'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'identity', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_resnet50/weights_best.pth', 'msf_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_resnet50/weights_best.pth', 'msf_cbam_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'softmax'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_mixstyle_resnet50/weights_best.pth', 'msf_cbam_mixstyle_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'identity', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/msf_cbam_edl_resnet50/weights_best.pth', 'msf_cbam_edl_resnet50'),
    ({'model': {'backbone': 'resnet50', 'feature_module': 'multiscale', 'attention': 'cbam', 'dg': 'mixstyle', 'head': 'evidential'}, 'dataset': {'num_classes': 4}}, 'outputs/trustoct_expB/weights_best.pth', 'TrustOCT (Proposed)')
]

for config_item, path, display_name in models_for_complexity:
    model_inst = build_model(config_item).to(device)
    if os.path.exists(path):
        model_inst.load_state_dict(torch.load(path, map_location=device))
        model_size_mb = os.path.getsize(path) / (1024 * 1024)
        size_str = f"{model_size_mb:.2f} MB"
    else:
        total_p = sum(p.numel() for p in model_inst.parameters())
        size_str = f"~{total_p * 4 / (1024*1024):.2f} MB"
        
    model_inst.eval()
    total_params = sum(p.numel() for p in model_inst.parameters())
    trainable_params = sum(p.numel() for p in model_inst.parameters() if p.requires_grad)
    
    with torch.no_grad():
        for _ in range(100):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        start_t = time.perf_counter()
        for _ in range(200):
            _ = model_inst(dummy_input)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end_t = time.perf_counter()
        
    avg_inf_ms = ((end_t - start_t) / 200) * 1000
    
    complexity_rows.append({
        'Model': display_name,
        'Total Params (M)': f"{total_params / 1e6:.2f}M",
        'Trainable Params (M)': f"{trainable_params / 1e6:.2f}M",
        'Size on Disk': size_str,
        'Inference Speed (ms)': f"{avg_inf_ms:.2f} ms"
    })

if len(complexity_rows) > 0:
    complexity_df = pd.DataFrame(complexity_rows)
    print('\n--- TABLE 6: COMPUTATIONAL COMPLEXITY ANALYSIS ---')
    display(complexity_df)
    os.makedirs('results/tables', exist_ok=True)
    complexity_df.to_csv('results/tables/table_6_computational_complexity.csv', index=False)


## Package & Download Results ZIP

In [ ]:
from google.colab import files
print('📦 Packaging all paper tables, figures, and model outputs...')

os.makedirs('results/tables', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

zip_filename = '/content/TrustOCT_Final_Paper_Results'
shutil.make_archive(zip_filename, 'zip', root_dir='.', base_dir='results')
zip_path = f'{zip_filename}.zip'
print(f'✅ Zip archive created: {zip_path}')

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    drive_backup = '/content/drive/MyDrive/TrustOCT_Results/TrustOCT_Final_Paper_Results.zip'
    os.makedirs(os.path.dirname(drive_backup), exist_ok=True)
    shutil.copy(zip_path, drive_backup)
    print(f'✅ Backup saved to Google Drive: {drive_backup}')
except Exception as e:
    print(f'Drive backup skipped: {e}')

print('\n⬇️ Downloading TrustOCT_Final_Paper_Results.zip to your computer...')
files.download(zip_path)
